# Notebook para Organização dos Dados e Chunking

### Integrantes

- Matteo Porcare - 10418276
- Tomy Goldberg Boimel - 10417109
- Yuri Milliet - 10417884
- Lucas Tuon de Matos - 10417987

### Conteúdo do Arquivo

Aqui está sendo desenvolvida a Organização dos Dados e Chunking. Ainda está incompleta, porém estamos trabalhando para completá-la.

## Bibliotecas

In [1]:
!pip install langchain-core langchain-text-splitters sentence-transformers faiss-cpu langchain-huggingface

In [2]:

import os
import pandas as pd 
import PyPDF2 
import re
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


## Procura os pdfs

In [3]:
# Adicionar aqui o caminho de onde os pdfs do dataset estão
caminho = r"C:\Users\Lucas\TRABALHOIA" 

# Lista apenas os arquivos que terminam com .pdf na pasta correta
arquivos_pdf = [f for f in os.listdir(caminho) if f.lower().endswith('.pdf')]

print(f"Encontrei {len(arquivos_pdf)} arquivos PDF na pasta TRABALHOIA:")
for pdf in arquivos_pdf:
    print(f"- {pdf}")

Encontrei 21 arquivos PDF na pasta TRABALHOIA:
- Artigo1-dualidadeIAeEDU_viesAlgoritmico.pdf
- Artigo2-estatisticaPrevencaoIA.pdf
- Artigo3-eticaNaEticaDaIA.pdf
- Aula1-Introdução.pdf
- Aula10-AnaliseExploratoria1.pdf
- Aula11-AnaliseExploratoria2.pdf
- Aula12-AprendizadodeMaquinaClassico1.pdf
- Aula13-AprendizadodeMaquinaClassico2.pdf
- Aula14-AprendizadodeMaquinaSupervisionado.pdf
- Aula15-RedesNeurais.pdf
- Aula16-CNNeVisaoComputacional.pdf
- Aula2-Etica.pdf
- Aula3-AgentesInteligentes.pdf
- Aula4-AgentesObjetivoseUtilidadade.pdf
- Aula5-AgentesLogicos.pdf
- Aula6-SistemasEspecialistas.pdf
- Aula7-DataPreparation1.pdf
- Aula8-DataPreparation2.pdf
- Aula9-DataPreparation3.pdf
- Livro-Inteligencia-Artificial-e-Sociedade-Conectada.pdf
- Livro1-ai2027.pdf


## Extração de dados

In [4]:
documentosExtraidos = []


caminhoBase = r"C:\Users\Lucas\TRABALHOIA"

for arquivo in arquivos_pdf:
    
    caminhoCompleto = os.path.join(caminhoBase, arquivo)
    
    with open(caminhoCompleto, 'rb') as f:
        leitorPdf = PyPDF2.PdfReader(f)
        textoCompleto = ""
        
        # Extrai o texto página por página
        for pagina in leitorPdf.pages:
            textoPagina = pagina.extract_text()
            if textoPagina:
                textoCompleto += textoPagina + "\n"
                
        documentosExtraidos.append({
            "nome_arquivo": arquivo,
            "conteudo": textoCompleto
        })

print("Extração concluída com sucesso!")


Extração concluída com sucesso!


In [5]:
# Transforma tudo em uma tabela e mostra as primeiras linhas
df_documentos = pd.DataFrame(documentosExtraidos)
display(df_documentos.head())

,nome_arquivo,conteudo
0,Artigo1-dualidadeIAeEDU_viesAlgoritmico.pdf,"Educ. Soc., Campinas, v. 46, e289323, 2025 1\n..."
1,Artigo2-estatisticaPrevencaoIA.pdf,"Revista Ibero -Americana de Humanidades, Ciên..."
2,Artigo3-eticaNaEticaDaIA.pdf,\n QUAL É O PAPEL DA ÉTICA NA ÉTICA DA INTELI...
3,Aula1-Introdução.pdf,Inteligência Artificial\nInteligência Artifici...
4,Aula10-AnaliseExploratoria1.pdf,Inteligência Artificial\nCompreensão e Análise...


## Tratamento dos Dados

In [6]:
# Função de Limpeza
def limpar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = re.sub(r'-\n+', '', texto)
    texto = re.sub(r'\n+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto)
    return texto.strip()

# Cria uma nova coluna para colocar o conteúdo limpo
df_documentos['conteudo_limpo'] = df_documentos['conteudo'].apply(limpar_texto)

# Função de Conversão para o formato LangChain
documentos_langchain = []
for index, row in df_documentos.iterrows():
    doc = Document(
        page_content=row['conteudo_limpo'],
        metadata={"source": row['nome_arquivo']} 
    )
    documentos_langchain.append(doc)

print(f"Total de PDFs processados: {len(documentos_langchain)}")

# Função de Configuração do Text Splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Geração dos Chunks
chunks = text_splitter.split_documents(documentos_langchain)

print(f"Total de chunks gerados para o FAISS: {len(chunks)}")
print("\n--- Amostra do Primeiro Chunk ---")
print(f"Fonte original: {chunks[0].metadata['source']}")
print(f"Tamanho do texto: {len(chunks[0].page_content)} caracteres")
print(f"Conteúdo: {chunks[0].page_content[:300]}...")

Total de PDFs processados: 21
Total de chunks gerados para o FAISS: 2322

--- Amostra do Primeiro Chunk ---
Fonte original: Artigo1-dualidadeIAeEDU_viesAlgoritmico.pdf
Tamanho do texto: 978 caracteres
Conteúdo: Educ. Soc., Campinas, v. 46, e289323, 2025 1 RESUMO: O estudo busca identificar as dualidades entre o uso da inteligência artificial (IA) na educação e os riscos de vieses algorítmicos. A pesquisa se insere como qualitativa e uma revisão sistemática de literatura com apoio bibliométrico. A coleta de...


## Embedding e FAISS
Ignorar possíveis erros "Error displaying widget: model not found"
O certo seria gerar uma barra de progresso, mas o Jupyter Notebook não tem esse suporte

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

import os

# Desliga a barra de progresso e os avisos por usar o jupyter
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

print("Baixando o modelo de embeddings...") # Pode demorar alguns segundos na primeira vez

# Carregar o modelo de Embeddings 
modelo_embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

print("Gerando vetores para os 2322 chunks e criando o banco FAISS. Aguarde...")

# Criar o banco de dados vetorial FAISS a partir dos chunks
banco_vetorial = FAISS.from_documents(chunks, modelo_embeddings) # Pode demorar minutos a depender do seu computador

print("Banco vetorial criado com sucesso!")

# Salvar o banco de dados localmente
caminho_banco = "./banco_faiss_ia"
banco_vetorial.save_local(caminho_banco)

print(f"Banco salvo na pasta: {caminho_banco}")

Baixando o modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Gerando vetores para os 2322 chunks e criando o banco FAISS. Aguarde...
Banco vetorial criado com sucesso!
Banco salvo na pasta: ./banco_faiss_ia
